# Train the solar bot’s language model — enhanced edition

This notebook teaches the bot **what your customers mean**: which intent a message is (price, fault, warranty, "what is MPPT"...), including the **solar terminology** (inverter, kVA, lithium vs gel, load shedding, changeover, Shona words...).

It does the whole job in one place:

| Step | What happens |
|---|---|
| 1 | Loads every example phrase the bot already has (starter set, Q&A sheet, or your live database) |
| 2 | Checks the data: thin intents, phrases that are in two intents at once |
| 3 | **Your terminology** (edit the glossary): synonyms, slang, Shona words, brand names, definition questions |
| 4 | Grows the training set from that glossary (variants, spelling mistakes, size formats like `5kva` / `5 kva`) |
| 5 | Measures honestly: every phrase is predicted by a model that never saw it |
| 6 | Tries several settings and picks the best |
| 7 | Shows what it still confuses, so you know what to teach next |
| 8 | Trains the final model, saves `models/nlu_model.pkl`, backs up the old one, and tests it |

**How to run:** open this file from inside the `solar-whatsapp-bot` folder with the project's `.venv` as the kernel, then *Run All*. The first full run takes a few minutes. After it finishes, deploy with `git add models/nlu_model.pkl`, then commit and push.

**To teach it more:** edit cell 3 (glossary, extra phrases, checks) and Run All again.

## 1. Settings

In [ ]:
import os, sys, re, json, random, time, tempfile, collections, logging, shutil
from pathlib import Path
import numpy as np

# ---------------------------------------------------------------- settings you may change
LIVE_DATABASE_URL = os.getenv("DATABASE_URL", "")  # "" = train on the starter phrases (seed + Q&A sheet).
#   For the live phrases staff added on the Train page, set the DATABASE_URL environment variable before starting
#   Jupyter (Render dashboard > giebee-db > Connections > External Database URL). Do not paste the URL into this
#   notebook if you share or commit it.
SEED = 7                  # makes every random step repeatable
N_FOLDS = 4               # how many parts the phrases are split into when measuring accuracy
AUG_PER_PHRASE = 3        # at most this many generated variants per real phrase
FULL_SEARCH = True        # True: try more settings (a few minutes longer); False: quick run
FORCE_CONFIG = None       # e.g. {"C": 50, "cw": "balanced"} to skip the search and use your own
OUT_FILE = Path(os.environ["NLU_OUT"]) if os.getenv("NLU_OUT") else None   # default: models/nlu_model.pkl

# ---------------------------------------------------------------- find the project
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nlu.py").exists()), None)
if PROJECT is None:
    raise SystemExit("Open this notebook from inside the solar-whatsapp-bot folder (the one that contains nlu.py).")
sys.path.insert(0, str(PROJECT)); os.chdir(PROJECT)
OUT_FILE = OUT_FILE or PROJECT / "models" / "nlu_model.pkl"

LIVE = bool(LIVE_DATABASE_URL)
os.environ["DATABASE_URL"] = LIVE_DATABASE_URL if LIVE else \
    "sqlite:///" + os.path.join(tempfile.mkdtemp(), "train.db").replace(os.sep, "/")
os.environ.setdefault("MARKET_AUTO", "0")
os.environ["NLU_ACCURACY_CHECK"] = "off"
logging.basicConfig(level=logging.WARNING)

import sklearn
pin = re.search(r"scikit-learn==([\d.]+)", (PROJECT / "requirements.txt").read_text())
print(f"Project: {PROJECT}")
print(f"scikit-learn here: {sklearn.__version__}   pinned for the server: {pin.group(1) if pin else '?'}")
if pin and pin.group(1) != sklearn.__version__:
    print("WARNING: different scikit-learn version. The server will refuse the model file. "
          f"Run:  pip install scikit-learn=={pin.group(1)}  and restart the kernel.")
print("Data source:", "LIVE database" if LIVE else "starter phrases (no DATABASE_URL set)")

## 2. Load the phrases and check them

In [ ]:
import brain, db, nlu
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

if not LIVE:
    brain.seed(json.loads((PROJECT / "catalog.json").read_text(encoding="utf-8")))
    brain._import_sheet()

intents = db.list_intents()
enabled = {n for n, i in intents.items() if i["enabled"]}
HANDLER = {n: i["handler"] for n, i in intents.items()}
real_raw = [(e["intent"], e["text"]) for e in db.list_examples() if e["intent"] in enabled]
if len(enabled) < 2 or len(real_raw) < 50:
    raise SystemExit(f"Only {len(real_raw)} phrases for {len(enabled)} intents found. Is DATABASE_URL the right database?")

# one row per (intent, cleaned text): this is exactly what the bot trains on
REAL = sorted({(i, nlu.clean(t)) for i, t in real_raw if nlu.clean(t)})
DIGEST = nlu.examples_digest(real_raw)       # lets the Train page tell whether staff changed phrases since

counts = collections.Counter(i for i, _ in REAL)
by_text = collections.defaultdict(set)
for i, t in REAL:
    by_text[t].add(i)
AMBIGUOUS = {t for t, s in by_text.items() if len(s) > 1}   # same words filed under two intents

print(f"{len(REAL)} phrases, {len(counts)} intents "
      f"(median {int(np.median(list(counts.values())))} phrases per intent)\n")
print("Thinnest intents (few examples = weak):")
for name, n in sorted(counts.items(), key=lambda kv: kv[1])[:12]:
    print(f"  {n:3d}  {name}")
print(f"\n{len(AMBIGUOUS)} phrases are filed under two intents at once (the model cannot get both right):")
for t in sorted(AMBIGUOUS)[:10]:
    print(f"  '{t}'  ->  {sorted(by_text[t])}")

## 3. Your terminology  (edit this cell to teach the model)

Three things you can change:

* **`GLOSSARY`**: groups of words that mean the same thing (English, slang, misspellings, Shona).
  * `terms`: the equivalent words. In every real phrase that contains one of them, the others are swapped in, so the bot learns they are the same.
  * `intent`: where a question *about* the term should go ("what is MPPT?", "explain lithium"). Use `None` if there is no matching intent.
  * `swap: False`: use for words that are related but **not** exact synonyms (they still get definition questions).
  * `templates` (optional): your own question patterns, with `{t}` where the term goes.
* **`EXTRA_PHRASES`**: real example sentences you want it to learn, per intent (the intent must exist).
* **`CHECKS`**: messages with the intent you expect. They are tested at the end, so you can see what the model got right.

The Shona entries come from phrases already in your data. Please have a Shona speaker confirm any you add.

In [ ]:
GLOSSARY = [
    # --- equipment ---
    {"name": "inverter", "terms": ["inverter", "invertor", "solar inverter", "power inverter"], "intent": None},
    {"name": "hybrid inverter", "terms": ["hybrid inverter", "hybrid", "hybrid system", "hybrid solar system",
                                          "all in one inverter", "all-in-one inverter"], "intent": "faq_inverter_types"},
    {"name": "off-grid", "terms": ["off grid", "offgrid", "off-grid", "standalone system", "stand alone system"],
     "intent": "qa_hybrid_vs_offgrid"},
    {"name": "grid-tied", "terms": ["grid tied", "grid-tie", "on grid", "ongrid", "net metering", "feed in",
                                    "sell power to zesa", "feed back to the grid"], "intent": "qa_grid_feed"},
    {"name": "mppt / charge controller", "terms": ["mppt", "mppt controller", "charge controller",
                                                     "solar charge controller", "solar controller", "pwm controller"],
     "intent": "faq_inverter_types"},
    {"name": "battery", "terms": ["battery", "batteries", "battery bank", "battery pack"], "intent": None},
    {"name": "lithium", "terms": ["lithium", "lithium ion", "lithium battery", "lifepo4", "lfp", "li-ion"],
     "intent": "qa_lithium_vs_gel"},
    {"name": "gel / lead acid", "terms": ["gel", "gel battery", "lead acid", "agm", "deep cycle", "tubular battery"],
     "intent": "qa_lithium_vs_gel"},
    {"name": "solar panel", "terms": ["solar panel", "panel", "pv panel", "pv module", "solar module",
                                      "mono panel", "monocrystalline panel"], "intent": None},
    {"name": "geyser", "terms": ["solar geyser", "geyser", "hot water", "water heater", "hot water system"],
     "intent": "faq_solar_geysers"},
    {"name": "borehole pump", "terms": ["borehole pump", "solar pump", "water pump", "submersible pump",
                                         "pumping system", "borehole"], "intent": "faq_solar_water_pumping"},
    {"name": "cctv", "terms": ["cctv", "security camera", "cameras", "surveillance"], "intent": "faq_cctv_security"},
    # --- sizes and units ---
    {"name": "kVA / kW", "terms": ["kva", "kw", "kilowatt", "kilowatts", "kilovolt amp"], "intent": "sizing"},
    {"name": "kWh", "terms": ["kwh", "kilowatt hour", "kilowatt hours"], "intent": "sizing"},
    # --- electrical protection (related, not exact synonyms: swap off) ---
    {"name": "breaker / protection", "terms": ["breaker", "circuit breaker", "mcb", "isolator", "dc isolator",
                                               "surge protector", "spd", "db board", "distribution board"],
     "intent": "faq_electrical_work", "swap": False},
    {"name": "earthing", "terms": ["earthing", "grounding", "earth leakage", "earth spike", "lightning protection"],
     "intent": "qa_earthing", "swap": False},
    {"name": "changeover", "terms": ["changeover", "change over", "ats", "automatic transfer switch",
                                      "transfer switch", "changeover switch"], "intent": "qa_changeover"},
    # --- power cuts (Zimbabwe) ---
    {"name": "power cut", "terms": ["power cut", "power outage", "load shedding", "loadshedding", "blackout",
                                     "zesa off", "no zesa", "zesa outage", "magetsi akadzima"],
     "intent": "faq_zesa_power_cuts"},
    {"name": "electricity", "terms": ["electricity", "power", "current", "magetsi"], "intent": None, "swap": False},
    # --- buying and paying ---
    {"name": "quotation", "terms": ["quote", "quotation", "estimate", "proforma", "proforma invoice", "pro forma"],
     "intent": "faq_whats_in_a_quote"},
    {"name": "deposit / instalments", "terms": ["deposit", "down payment", "part payment", "instalments", "installments",
                                                 "layby", "lay by", "payment plan", "pay in parts"],
     "intent": "faq_payment_deposit"},
    {"name": "warranty", "terms": ["warranty", "guarantee", "guarantee period"], "intent": "faq_warranty"},
    {"name": "site visit", "terms": ["site visit", "site survey", "site inspection", "technician visit",
                                      "assessment"], "intent": "faq_site_visit"},
    {"name": "installation", "terms": ["installation", "install", "installing", "fitting", "setup", "commissioning"],
     "intent": None},
    {"name": "payback", "terms": ["payback", "payback period", "roi", "return on investment", "break even"],
     "intent": "faq_savings_payback", "swap": False},
    {"name": "monitoring", "terms": ["monitoring", "remote monitoring", "wifi monitoring", "solarman", "smart app"],
     "intent": "qa_monitoring", "swap": False},
    # --- faults ---
    {"name": "not working", "terms": ["not working", "stopped working", "has packed up", "gone bad", "is dead",
                                       "has failed"], "intent": None},
    # --- brands (questions like "do you sell Deye") ---
    {"name": "brands", "terms": ["growatt", "deye", "victron", "sunsynk", "felicity", "jinko", "longi",
                                  "canadian solar", "pylontech"], "intent": "faq_brands", "swap": True,
     "templates": ["do you sell {t}", "do you have {t}", "is {t} good", "{t} price list", "are you a {t} dealer",
                   "which {t} do you stock"]},
]

# The default question patterns for a term whose group has an intent
DEFINITION_TEMPLATES = ["what is {t}", "what is a {t}", "what does {t} mean", "explain {t}", "explain {t} to me",
                        "tell me about {t}", "i dont understand {t}", "{t} meaning", "how does {t} work",
                        "{t}", "can you explain {t}"]

# Sizes get their own question patterns (kVA / kW written both ways)
SIZES = ["1.5kva", "2kva", "3kva", "3.5kva", "5kva", "5.5kva", "8kva", "10kva", "12kva", "15kva", "20kva",
         "3kw", "5kw", "8kw", "10kw"]
SIZE_TEMPLATES = {
    "price": ["how much is a {s} system", "price of {s} solar", "marii {s}", "{s} price", "how much for {s}",
              "what does {s} cost", "cost of a {s} inverter system", "{s} solar kit price"],
    "buy_system": ["i want a {s} system", "i need a {s} solar system installed", "quote for {s}",
                   "can you quote me a {s} system", "i am looking for a {s} hybrid system", "ndoda {s} solar"],
    "sizing": ["what can a {s} system run", "will a {s} run my fridge and tv", "is {s} enough for my house",
               "can a {s} inverter run a borehole pump"],
}

# Real sentences you want it to learn: {"intent_name": ["phrase", ...]}
EXTRA_PHRASES = {
    # "faq_inverter_types": ["is a hybrid better than an ordinary inverter", "what is the difference between mppt and pwm"],
    # "maintenance_fault": ["inverter showing e03", "my inverter keeps going into overload"],
}

# Messages and what the bot should understand them as. Tested at the end.
CHECKS = [
    ("what is mppt", "faq_inverter_types"),
    ("whats the difference between lithium and gel", "qa_lithium_vs_gel"),
    ("do you sell deye inverters", "faq_brands"),
    ("marii 5kva", "price"),
    ("my inverter keeps beeping and the battery wont charge", "maintenance_fault"),
    ("can i pay a deposit and the balance in instalments", "faq_payment_deposit"),
    ("i want a quotation for 5kva hybrid", "buy_system"),
    ("what is load shedding", "faq_zesa_power_cuts"),
    ("is there smoke coming from my inverter", "qa_overheating"),
    ("hello", "greeting"),
    ("thank you", "thanks"),
    ("can i talk to a person", "talk_to_human"),
    ("do you offer a warranty", "faq_warranty"),
    ("what is an earth leakage", "qa_earthing"),
    ("i need a changeover switch", "qa_changeover"),
    ("can solar pump water from my borehole", "faq_solar_water_pumping"),
    ("how long does installation take", "faq_installation_time"),
    ("where are you located", "contact_info"),
]

# Things that are NOT about solar: the bot should say "not understood" (or off_topic), never guess an intent
OUT_OF_SCOPE = ["asdfghjkl", "qwerty zxcv", "who won the soccer match yesterday", "what is the capital of france",
                "lorem ipsum dolor sit amet", "12345678", "tell me a joke about cats", "how do i bake a cake"]

# ---- validation
bad = sorted({g["intent"] for g in GLOSSARY if g["intent"] and g["intent"] not in enabled} |
             {i for i in EXTRA_PHRASES if i not in enabled} |
             {i for i in SIZE_TEMPLATES if i not in enabled} |
             {w for _, w in CHECKS if w not in enabled})
if bad:
    print("These intents do not exist in the data, so their phrases/checks will not work:", bad)
print(f"{len(GLOSSARY)} glossary groups, {sum(len(g['terms']) for g in GLOSSARY)} terms, "
      f"{sum(len(v) for v in EXTRA_PHRASES.values())} extra phrases, {len(CHECKS)} checks")

## Additional training data and knowledge-base preparation

The companion `solar_training_examples.csv` contains **draft synthetic phrases**, not verified customer messages. Review the intent labels and Shona phrasing with staff before using them. These phrases must **not** be included in a held-out evaluation set. Only intents already enabled in your database will be imported.


In [ ]:
# Optional: load a reviewed CSV of extra examples BEFORE cell 4 training-data generation.
# Run this cell after the glossary cell (cell 6) and before the augmentation cell (cell 8).
import csv
from pathlib import Path
CSV_FILE = PROJECT / "solar_training_examples.csv"
if not CSV_FILE.exists():
    CSV_FILE = Path("solar_training_examples.csv")
if CSV_FILE.exists():
    accepted = []
    skipped = collections.Counter()
    with CSV_FILE.open(encoding="utf-8-sig", newline="") as handle:
        for row in csv.DictReader(handle):
            intent = (row.get("intent") or "").strip()
            phrase = (row.get("text") or "").strip()
            reviewed = (row.get("reviewed") or "").strip().lower()
            if reviewed not in {"yes", "true", "1"}:
                skipped["not_reviewed"] += 1
            elif intent not in enabled:
                skipped["unknown_intent"] += 1
            elif not phrase:
                skipped["empty"] += 1
            else:
                accepted.append((intent, phrase))
    for intent, phrase in accepted:
        EXTRA_PHRASES.setdefault(intent, []).append(phrase)
    print(f"Imported {len(accepted)} reviewed examples; skipped: {dict(skipped)}")
else:
    print("Optional CSV not found; using existing notebook phrases only")


## 4. Grow the training set from the glossary

Four kinds of generated phrase (none are ever used to judge accuracy, so the score stays honest):

1. **Definition questions**: "what is mppt", "explain lithium to me" for every glossary term that has an intent.
2. **Size questions**: "how much is a 5kva system", "marii 8kw", "ndoda 10kva solar".
3. **Variants**: a real phrase with a term swapped for its synonyms, and `5kva` written as `5 kva` or `5kw`.
4. **Spelling mistakes**: one letter dropped, doubled or swapped in a word (the bot already fixes common SMS shorthand like `pls`, so those are not generated).

In [ ]:
def _bounded(term):
    return re.compile(r"(?<![a-z0-9])" + re.escape(term) + r"(?![a-z0-9])")

# glossary groups with cleaned terms (the bot cleans every message first, e.g. "panel" -> "panels")
GROUPS = []
for g in GLOSSARY:
    terms = list(dict.fromkeys(t for t in (nlu.clean(x) for x in g["terms"]) if t))
    GROUPS.append({**g, "clean": terms, "res": {t: _bounded(t) for t in terms}})

def template_phrases():
    out = []
    for g in GROUPS:
        if not g["intent"] or g["intent"] not in enabled:
            continue
        for t in g["clean"]:
            for tpl in g.get("templates") or DEFINITION_TEMPLATES:
                out.append((g["intent"], nlu.clean(tpl.format(t=t))))
    for intent, tpls in SIZE_TEMPLATES.items():
        if intent in enabled:
            for s in SIZES:
                out += [(intent, nlu.clean(tpl.format(s=s))) for tpl in tpls]
    return out

TEMPLATE_ROWS = sorted(set(template_phrases()))
EXTRA_ROWS = sorted({(i, nlu.clean(p)) for i, ps in EXTRA_PHRASES.items() if i in enabled for p in ps if nlu.clean(p)})

def synonym_variants(text):
    out = []
    for g in GROUPS:
        if not g.get("swap", True):
            continue
        for term in g["clean"]:
            if g["res"][term].search(text):
                out += [g["res"][term].sub(other, text, count=1) for other in g["clean"] if other != term]
                break
    return out

UNIT = re.compile(r"(\d+(?:\.\d+)?)\s*(kva|kw)\b")
def unit_variants(text):
    m = UNIT.search(text)
    if not m:
        return []
    n, u = m.groups()
    other = "kw" if u == "kva" else "kva"
    return [text[:m.start()] + f + text[m.end():] for f in (f"{n} {u}", f"{n}{other}", f"{n} {other}")]

def typo_variant(text, rng):
    words = text.split()
    idx = [k for k, w in enumerate(words) if len(w) >= 5 and w.isalpha()]
    if not idx:
        return None
    k = rng.choice(idx); w = words[k]; p = rng.randrange(1, len(w) - 1)
    op = rng.choice(("drop", "double", "swap"))
    words[k] = w[:p] + w[p + 1:] if op == "drop" else w[:p] + w[p] + w[p:] if op == "double" else w[:p] + w[p + 1] + w[p] + w[p + 2:]
    return " ".join(words)

def augment_phrase(text, rng, limit=AUG_PER_PHRASE):
    pool = synonym_variants(text) + unit_variants(text)
    rng.shuffle(pool)
    out = pool[:limit]
    if len(out) < limit:
        t = typo_variant(out[0] if out and rng.random() < .3 else text, rng)
        if t:
            out.append(t)
    return [c for c in (nlu.clean(o) for o in out) if c and c != text]

demo_rng = random.Random(SEED)
print(f"{len(TEMPLATE_ROWS)} definition/size phrases, {len(EXTRA_ROWS)} of your own phrases\n")
touched = [r for r in REAL if UNIT.search(r[1]) or any(g["res"][t].search(r[1]) for g in GROUPS for t in g["clean"])]
for intent, text in random.Random(1).sample(touched, min(6, len(touched))):
    print(f"[{intent}] {text}")
    for v in augment_phrase(text, demo_rng):
        print(f"      -> {v}")

## 5. Measure it honestly

`cross_validate` splits the real phrases into parts. For each part, the model is trained on the *other* parts (plus generated phrases made only from those) and then has to predict the held-out part. Reported:

* **accuracy / macro-F1**: exact intent. Macro-F1 gives a small intent the same weight as a big one.
* **top-3**: the right intent is among the model's first three guesses.
* **same-handler accuracy**: the guess may be a different intent, but one the bot treats the same way (e.g. two FAQ replies). A wrong *handler* is what actually sends a customer down the wrong path.
* **coverage**: how often the bot is sure enough to answer instead of asking again; **precision** is how often it is right when it answers.

In [ ]:
LABELS = [i for i, _ in REAL]
TEXTS = [t for _, t in REAL]
KEEP = np.array([t not in AMBIGUOUS for t in TEXTS])          # ambiguous phrases are trained on but not scored

def make_training(train_idx, test_texts, cfg, rng):
    rows = [REAL[k] for k in train_idx]
    seen = set(rows)
    synth = []
    if cfg["gloss"]:
        synth += TEMPLATE_ROWS + EXTRA_ROWS
    if cfg["aug"]:
        for intent, text in rows:
            synth += [(intent, v) for v in augment_phrase(text, rng)]
    for r in synth:
        if r[1] in test_texts or r in seen:      # never let a generated phrase equal a held-out one
            continue
        seen.add(r); rows.append(r)
    return rows

def fit_model(rows, cfg):
    feat = nlu._Tfidf()
    x = feat.fit([t for _, t in rows])
    y = np.array([i for i, _ in rows])
    clf = LogisticRegression(C=cfg["C"], max_iter=400, tol=1e-3, class_weight=cfg["cw"])
    clf.fit(x, y)
    return feat, clf, x, y

def cross_validate(cfg, folds=N_FOLDS, verbose=True):
    t0 = time.time()
    y_all = np.array(LABELS)
    folds = int(min(folds, min(collections.Counter(LABELS).values())))
    skf = StratifiedKFold(folds, shuffle=True, random_state=SEED)
    true, pred, top3, understood = [], [], [], []
    for n, (tr, te) in enumerate(skf.split(np.zeros(len(y_all)), y_all), 1):
        te = [k for k in te if KEEP[k]]
        test_texts = {TEXTS[k] for k in te}
        rows = make_training(tr, test_texts, cfg, random.Random(SEED + n))
        feat, clf, xtr, ytr = fit_model(rows, cfg)
        xte = feat.transform([TEXTS[k] for k in te])
        proba = clf.predict_proba(xte)
        sim = (xte @ xtr.T).toarray().max(axis=1)
        order = np.argsort(-proba, axis=1)
        for r, k in enumerate(te):
            ranked = [(str(clf.classes_[j]), float(proba[r, j])) for j in order[r, :3]]
            top3.append(LABELS[k] in {c for c, _ in ranked})
            res = {"intent": ranked[0][0], "confidence": ranked[0][1], "similarity": float(sim[r]),
                   "ranked": ranked, "backend": "tfidf"}
            understood.append(nlu.engine.understood(res))
            true.append(LABELS[k]); pred.append(ranked[0][0])
    true, pred, und = np.array(true), np.array(pred), np.array(understood)
    ok = true == pred
    m = {"accuracy": ok.mean(), "macro_f1": f1_score(true, pred, average="macro", zero_division=0),
         "top3": float(np.mean(top3)),
         "same_handler": float(np.mean([HANDLER[a] == HANDLER[b] for a, b in zip(true, pred)])),
         "coverage": und.mean(), "precision": ok[und].mean() if und.any() else 0.0,
         "true": true, "pred": pred, "seconds": time.time() - t0}
    if verbose:
        print(f"  accuracy {m['accuracy']:.3f}  macro-F1 {m['macro_f1']:.3f}  top-3 {m['top3']:.3f}  "
              f"same-handler {m['same_handler']:.3f}  coverage {m['coverage']:.3f}  precision {m['precision']:.3f}  "
              f"({m['seconds']:.0f}s)")
    return m

BASELINE = {"name": "A. as the bot trains today", "C": 20, "cw": "balanced", "aug": False, "gloss": False}
print(BASELINE["name"]); base = cross_validate(BASELINE)

## 6. Try settings and pick the best

In [ ]:
CONFIGS = [
    {"name": "B. + your terminology", "C": 20, "cw": "balanced", "aug": False, "gloss": True},
    {"name": "C. + variants and spelling mistakes", "C": 20, "cw": "balanced", "aug": True, "gloss": True},
]
if FULL_SEARCH:
    CONFIGS += [
        {"name": "D. C=50", "C": 50, "cw": "balanced", "aug": True, "gloss": True},
        {"name": "E. C=100", "C": 100, "cw": "balanced", "aug": True, "gloss": True},
        {"name": "F. C=50, no class weights", "C": 50, "cw": None, "aug": True, "gloss": True},
        {"name": "G. C=200", "C": 200, "cw": "balanced", "aug": True, "gloss": True},
    ]
if FORCE_CONFIG:
    CONFIGS = [{"name": "forced", "aug": True, "gloss": True, **FORCE_CONFIG}]

RESULTS = [(BASELINE, base)]
for cfg in CONFIGS:
    print(cfg["name"])
    RESULTS.append((cfg, cross_validate(cfg)))

# best = highest macro-F1 (ties: higher same-handler accuracy)
best_cfg, best = max(RESULTS, key=lambda r: (round(r[1]["macro_f1"], 4), r[1]["same_handler"]))
print("\n{:<42}{:>9}{:>9}{:>8}{:>9}".format("setting", "accuracy", "macro-F1", "top-3", "handler"))
for cfg, m in RESULTS:
    mark = "  <== chosen" if cfg is best_cfg else ""
    print("{:<42}{:>9.3f}{:>9.3f}{:>8.3f}{:>9.3f}{}".format(cfg["name"], m["accuracy"], m["macro_f1"], m["top3"],
                                                            m["same_handler"], mark))
print(f"\nGain over today's model: accuracy {base['accuracy']:.3f} -> {best['accuracy']:.3f}, "
      f"same-handler {base['same_handler']:.3f} -> {best['same_handler']:.3f}")

## 7. What it still gets wrong

Two lists. **Intents it misses most** show where to add more example phrases. **Confused pairs** show intents that overlap; if the pair has the *same handler* the customer still gets a sensible answer, but if the handlers differ it is worth fixing (add distinguishing phrases in cell 3, or merge the two intents on the Train page).

In [ ]:
true, pred = best["true"], best["pred"]
rows = []
for name in sorted(set(true)):
    mask = true == name
    rows.append((float((pred[mask] == name).mean()), int(mask.sum()), name))
print("Intents it misses most (recall, phrases tested):")
for r, n, name in sorted(rows)[:15]:
    wrong = collections.Counter(pred[(true == name) & (pred != name)]).most_common(2)
    print(f"  {r:4.0%}  {n:3d}  {name:<28} often read as: {', '.join(f'{w} x{c}' for w, c in wrong)}")

pairs = collections.Counter((a, b) for a, b in zip(true, pred) if a != b)
print("\nMost confused pairs (true -> predicted):")
for (a, b), c in pairs.most_common(15):
    flag = "same handler, harmless" if HANDLER[a] == HANDLER[b] else f"DIFFERENT handler ({HANDLER[a]} vs {HANDLER[b]})"
    print(f"  x{c:<3} {a:<26} -> {b:<26} {flag}")

## 8. Train the final model, save it, test it

Trains on **all** real phrases plus everything generated, with the chosen settings. The old model file is copied next to it with a timestamp before it is replaced. The file has the exact layout `nlu.py` expects, so the bot loads it with no code change.

In [ ]:
cfg = best_cfg
rng = random.Random(SEED)
rows = make_training(range(len(REAL)), set(), cfg, rng)
print(f"Training on {len(rows)} phrases ({len(REAL)} real + {len(rows) - len(REAL)} generated), "
      f"{len({i for i, _ in rows})} intents, setting: {cfg['name']}")
t0 = time.time()
feat, clf, x, y = fit_model(rows, cfg)
print(f"trained in {time.time() - t0:.0f}s")

model = {"feat": feat, "clf": clf, "x": x, "y": y, "texts": [t for _, t in rows]}
eng = nlu.engine
with eng._lock:
    eng._model = eng._quick = model
    eng.info = {"trained": True, "backend": "tfidf", "examples": len(rows), "intents": len({i for i, _ in rows}),
                "accuracy": round(float(best["accuracy"]), 3), "note": "",
                "model": "scikit-learn TF-IDF + logistic regression"}

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
PREVIOUS = None
if OUT_FILE.exists():
    backup = OUT_FILE.with_name(f"{OUT_FILE.stem}.backup-{time.strftime('%Y%m%d-%H%M%S')}{OUT_FILE.suffix}")
    shutil.copy2(OUT_FILE, backup)
    PREVIOUS = backup
    print("Old model kept as", backup.name)
meta = eng.save(OUT_FILE, digest=DIGEST)
print(f"Saved {OUT_FILE} ({OUT_FILE.stat().st_size / 1024:.0f} KB): {meta['examples']} phrases, "
      f"{meta['intents']} intents, scikit-learn {meta['sklearn']}")

report = {"trained_at": time.strftime("%Y-%m-%d %H:%M:%S"), "data": "live database" if LIVE else "starter phrases",
          "real_phrases": len(REAL), "generated_phrases": len(rows) - len(REAL), "setting": cfg["name"],
          "held_out": {k: round(float(best[k]), 3) for k in ("accuracy", "macro_f1", "top3", "same_handler", "coverage", "precision")},
          "today_model_held_out": {k: round(float(base[k]), 3) for k in ("accuracy", "macro_f1", "top3", "same_handler")}}
(OUT_FILE.parent / "nlu_train_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report["held_out"]))

### Test the saved file the way the server will use it

A fresh engine loads the file, then your `CHECKS` (cell 3) and the out-of-scope messages run through it.

In [ ]:
fresh = nlu.Engine()
info = fresh.load(OUT_FILE, digest=DIGEST)
print("Loaded OK, in sync with the phrases:", info["in_sync"], "|", info["examples"], "phrases,", info["intents"], "intents\n")

def read(text):
    r = fresh.predict(text)
    return r, fresh.understood(r)

def run_checks(engine):
    out = []
    for text, want in CHECKS:
        r = engine.predict(text)
        out.append(r["intent"] == want and engine.understood(r))
    return out

old_score = None
if PREVIOUS is not None:
    try:
        old = nlu.Engine(); old.load(PREVIOUS)
        old_score = sum(run_checks(old))
    except Exception as exc:
        print("(could not load the previous model for comparison:", exc, ")")

passed = 0
for text, want in CHECKS:
    r, ok = read(text)
    good = r["intent"] == want and ok
    passed += good
    alt = "" if good else f"   expected {want}"
    print(f"{'PASS' if good else 'FAIL'}  {text!r:<58} -> {r['intent']} ({r['confidence']:.0%}{'' if ok else ', NOT SURE'}){alt}")
print(f"\n{passed}/{len(CHECKS)} checks passed" +
      (f"   (the previous model passed {old_score}/{len(CHECKS)})" if old_score is not None else ""))

print("\nOut of scope (should be 'not sure' or off_topic):")
leaks = 0
for text in OUT_OF_SCOPE:
    r, ok = read(text)
    bad = ok and r["intent"] != "off_topic"
    leaks += bad
    print(f"{'LEAK' if bad else 'ok  '}  {text!r:<45} -> {r['intent']} ({r['confidence']:.0%}{'' if ok else ', not sure'})")
print(f"{leaks}/{len(OUT_OF_SCOPE)} guessed an intent for something unrelated")

### Try your own messages

In [ ]:
def ask(text):
    r, ok = read(text)
    print(f"{text!r}\n  intent: {r['intent']}  confidence: {r['confidence']:.0%}  understood: {ok}")
    print("  top 3:", ", ".join(f"{n} {p:.0%}" for n, p in r["ranked"]))
    print("  closest taught phrase:", r["matched"])
    print("  details found:", nlu.extract_entities(text))

for m in ["how much is a 5kva hybrid system with lithium batteries", "magetsi akadzima, mungandibatsira here",
          "does the mppt controller come with the inverter"]:
    ask(m); print()
# ask("type your own message here")

## Solar books, manuals and documents: a separate knowledge base

**Why not train on them.** This model only decides *what a customer means* (price, fault, warranty...). It cannot learn facts from a long document, and pasting a book into `EXTRA_PHRASES` would just teach it odd phrases. Facts belong in a knowledge base that the bot looks things up in.

### What your bot already has
| Layer | Where | What it does |
|---|---|---|
| Intent model | this notebook, `nlu.py` | Works out what the customer wants |
| Exact lookups | `knowledge.py`, `data/faults.json` | Inverter fault codes by brand ("F56", "error 08") and short Zimbabwe regulation facts. A code that is not in the table is reported as unknown and handed to a technician, never guessed |
| Grounded answers | `llm.answer_from_facts` | The AI step may only reword facts the bot gives it |
| Business facts | `catalog.json`, `quote_book.json` | Prices, packages, hours, branches |

A document knowledge base would be a fourth source of *facts* feeding the grounded-answer step. It does not replace the intent model or the exact lookups.

### Build order (start where the pay-off is biggest)
1. **Fault codes for the brands you actually sell.** Extend `data/faults.json` from the manuals of the exact models you install. A table is more accurate than search for codes and error numbers, and it already works.
2. **Approved FAQ answers.** Turn the questions staff answer most into reviewed entries (this is what `catalog.json` FAQs and the Q&A sheet already do).
3. **Searchable manuals, only if 1 and 2 leave real gaps.** Index licensed PDFs, split into small passages, and fetch the few best passages when a question needs a technical fact.

### If you do build step 3
* **Keep with every passage:** document title, manufacturer, model number, revision date, page and source URL. Show the source when you answer, and prefer the newest revision.
* **Only the exact model.** A Deye 5 kW answer must not be given for a Growatt. If the customer's model is unknown, ask for it or hand over to a technician.
* **No answer found means no answer.** Below a relevance threshold the bot says it will check with a technician. It must not fill the gap from memory.
* **Retrieved text is data, never instructions.** Strip or ignore anything in a document that reads like a command to the bot.
* **Licensing.** Index only material you have the right to store: your own manuals and datasheets, supplier documents you are allowed to use, and openly licensed or public sources. Do not copy textbooks or paywalled PDFs. For books such as *Photovoltaic Systems* (Dunlop) or the *Solar Electricity Handbook* (Boxwell), use authorised copies for staff reference, not for the bot.
* **Check it before it goes live.** Write 30 to 50 real technical questions with the expected answer and source, and measure how often the bot finds the right passage and how often it wrongly answers when it should have deferred.

### Where to collect material
| Need | Source |
|---|---|
| Fault codes, wiring, settings for what you sell | The manufacturers' manuals for your exact models (Deye, Growatt, Victron, Sunsynk, Felicity...) |
| Battery and charging theory | Victron Energy technical documentation; the battery maker's datasheet |
| Sun, yield and system design | NREL and U.S. DOE solar education pages; Global Solar Atlas |
| Zimbabwe rules and tariffs | ZERA and ZETDC/ZESA official notices (check the date: rules change) |

### Never from a document
Do not take **prices, stock, warranty terms or installation promises** from manuals or the web. They come from your live catalogue (`catalog.json`, `quote_book.json`) or from staff.

### Safety comes first
For smoke, burning smells, fire, sparks, shocks, swollen batteries, exposed wiring or water in the equipment, the bot gives a short safety message (switch off if it is safe to do so, keep away, do not open or repair it) and passes the customer to a qualified technician. It never talks someone through a repair. This is already how the `qa_overheating` and `qa_water_damage` intents are handled, so keep their example phrases well stocked (see the thinnest-intents list in section 2).

## Deploy

```
git add models/nlu_model.pkl models/nlu_train_report.json
git commit -m "Retrained model"
git push
```

Render redeploys and the bot loads the new file. The ERP **Train** page shows whether staff changed phrases after this model was trained; if it says so, run this notebook again (with `DATABASE_URL` set to read the live phrases).

**Remember:** phrases you add in cell 3 live in this notebook and in the model file, not in the database. Keep the notebook, or add the important ones on the Train page too.